# Docling quickstart

The fastest way to see what Docling does: no server, no pipeline, no cluster — just the Python SDK running in this workbench. This notebook converts a PDF, an image, and a live web page to Markdown and JSON, then chunks a document for downstream RAG use.

Run this in an OpenShift AI workbench (Standard Data Science or PyTorch image) after installing the requirements below.

In [ ]:
%pip install -q -r ../requirements.txt

## 1. Convert a PDF to Markdown and JSON

`DocumentConverter` auto-detects the format from the file extension/content and runs the right pipeline (layout model + OCR for scanned pages, native text extraction for born-digital PDFs).

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
result = converter.convert("../../samples/structured.pdf")

markdown = result.document.export_to_markdown()
print(markdown[:1500])

In [ ]:
doc_dict = result.document.export_to_dict()
print(f"Top-level keys: {list(doc_dict.keys())}")
print(f"Pages: {len(result.document.pages)}")

## 2. Scanned PDF: same call, OCR happens automatically

`scanned.pdf` has no embedded text layer. Docling detects that and routes the pages through OCR — no extra code on your end.

In [ ]:
scanned_result = converter.convert("../../samples/scanned.pdf")
print(scanned_result.document.export_to_markdown()[:1000])

## 3. Convert an image

Docling treats a standalone image (PNG/JPEG/TIFF) the same way it treats a scanned PDF page: layout analysis + OCR straight to structured Markdown/JSON. Swap in your own image path below.

In [ ]:
# image_result = converter.convert("path/to/your/image.png")
# print(image_result.document.export_to_markdown())

## 4. Convert a live web page

Pass a URL directly — Docling fetches and parses the HTML itself.

In [ ]:
web_result = converter.convert("https://docling-project.github.io/docling/")
print(web_result.document.export_to_markdown()[:1500])

## 5. Chunk a document for RAG

`HybridChunker` splits on document structure (headings, sections, tables) instead of a fixed character count, so each chunk stays semantically coherent. This is the input shape the `rag-via-ogx-example/` and `batch-via-pipeline-example/` quickstarts build on.

In [ ]:
from docling.chunking import HybridChunker

chunker = HybridChunker()
chunks = list(chunker.chunk(dl_doc=result.document))

print(f"{len(chunks)} chunks")
for chunk in chunks[:3]:
    print("---")
    print(chunk.text[:300])

## Next steps

- Need this behind a REST API instead of inline SDK calls? See `docling-serve/` + `docling-serve-examples/`.
- Converting thousands of documents at once? See `batch-via-pipeline-example/`.
- Want the chunks in a vector database for RAG? See `rag-via-ogx-example/`.